# Go2 ODD/COD Observer - 10-Agent Workflow

**Autonomous ODD Compliance Analysis for Quadruped Robots**

This notebook demonstrates a complete multi-agent AI system that analyzes robot sensor data to determine if a robot is operating within its **Operational Design Domain (ODD)**.

## Key Concepts

- **ODD (Operational Design Domain)**: The environment the robot is **designed** to work in (specification)
- **COD (Current Operating Domain)**: The environment the robot is **actually** in (measured from sensors)
- **ODD Compliance**: Comparison of COD against ODD to detect violations and safety boundaries

## 10-Agent Sequential Pipeline

1. **ODD Spec Agent** → Convert natural language ODD to formal specification (runs FIRST)
2. **Perception Loop + Summary** → Analyze camera + LiDAR images, classify sim vs real
3. **Motion Loop + Summary** → IMU-based motion detection (accelerometer/gyroscope)
4. **Collision Loop + Summary** → Multimodal fusion for risk assessment
5. **COD Classifier** → Classify current operating domain from sensor data
6. **ODD Compliance** → Compare COD vs ODD, detect violations
7. **Report Generator** → Create comprehensive human-readable report

## What You'll Need

- Google Gemini API key (free at https://aistudio.google.com/app/apikey)
- Preprocessed scenario data (from `extract_windows.py` script)

Let's get started! 🚀

In [ ]:
import os
import sys
import json
import asyncio
from pathlib import Path
from dotenv import load_dotenv

# Add scripts directory to path to import our workflow
sys.path.insert(0, str(Path().absolute().parent / "scripts"))

# Load environment variables
load_dotenv()

# Check for API key
GOOGLE_API_KEY = os.getenv('GOOGLE_API_KEY')
if not GOOGLE_API_KEY:
    print("❌ GOOGLE_API_KEY not found!")
    print()
    print("Please set your API key:")
    print("  1. Get free key: https://aistudio.google.com/app/apikey")
    print("  2. Set environment: export GOOGLE_API_KEY='your-key-here'")
    print("  3. Or create .env file with: GOOGLE_API_KEY=your-key-here")
else:
    print("✅ Google Gemini API configured")
    print(f"   Key: {GOOGLE_API_KEY[:8]}...{GOOGLE_API_KEY[-4:]}")

# Import visualization libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Libraries imported successfully")

In [ ]:
# Natural Language ODD Description
# Customize this for your robot's design constraints

nl_odd_description = """
The Unitree Go2 quadruped robot is designed for indoor navigation in office environments.

OPERATIONAL CONSTRAINTS:

1. Environment Type:
   - Designed for: indoor_office, indoor_corridor
   - Prohibited: outdoor environments, staircases, unstructured terrain

2. Lighting Conditions:
   - Required: bright or dim lighting (adequate visibility)
   - Prohibited: dark environments (requires vision sensors)

3. Terrain Type:
   - Designed for: smooth_floor (tile, hardwood, low-pile carpet)
   - Prohibited: rough or very_rough terrain, stairs, slopes >10°

4. Speed Range:
   - Normal operation: 0.0 to 1.5 m/s
   - Physical limit: 2.5 m/s (emergency only)

5. Obstacle Density:
   - Acceptable: low to moderate obstacles (0.0 to 0.6 normalized)
   - Boundary: 0.6 to 0.8 (crowded but navigable)
   - Prohibited: >0.8 (too cluttered for safe navigation)

6. Traversability:
   - Required: navigable space (0.5 to 1.0 score)
   - Boundary: 0.3 to 0.5 (challenging but possible)
   - Prohibited: <0.3 (impassable or unsafe)

7. Collision Risk:
   - Acceptable: low risk (0.0 to 0.3 likelihood)
   - Boundary: 0.3 to 0.5 (caution required)
   - Prohibited: >0.5 (high risk, stop immediately)

8. Platform Stability:
   - Required: stable platform (roll/pitch <15°)
   - Boundary: 15° to 20° (unstable but recoverable)
   - Prohibited: >20° (tip-over risk)
"""

print(f"✅ ODD specification defined ({len(nl_odd_description)} characters)")
print("\nYou can customize this description for different robot types:")
print("  • Outdoor delivery robots")
print("  • Aerial drones (altitude, wind, GPS)")
print("  • Warehouse AMRs (floor type, shelf proximity)")
print("  • Autonomous vehicles (road type, traffic, weather)")

In [ ]:
# Available scenarios
data_dir = Path("../data/processed/runs")

if data_dir.exists():
    available_scenarios = [d.name for d in data_dir.iterdir() if d.is_dir()]
    print("📁 Available scenarios:")
    for i, scenario in enumerate(available_scenarios, 1):
        window_count = len(list((data_dir / scenario).glob("motion_*.json")))
        print(f"   {i}. {scenario} ({window_count} windows)")
else:
    print("⚠️ Data directory not found. Please run extract_windows.py first.")
    available_scenarios = []

# Select scenario
SCENARIO_NAME = "sim_run_test"  # Change this to your scenario
print(f"\n✅ Selected scenario: {SCENARIO_NAME}")

# Verify scenario exists
scenario_path = data_dir / SCENARIO_NAME
if not scenario_path.exists():
    print(f"❌ Scenario not found: {scenario_path}")
    print("   Please update SCENARIO_NAME to match an available scenario")
else:
    window_count = len(list(scenario_path.glob("motion_*.json")))
    print(f"   Path: {scenario_path}")
    print(f"   Windows: {window_count}")

In [ ]:
# Import the workflow function
from odd_workflow_full import run_odd_workflow

# Run the complete analysis
print("🚀 Starting 10-agent ODD/COD analysis workflow...")
print(f"   Scenario: {SCENARIO_NAME}")
print(f"   ODD: {len(nl_odd_description)} chars")
print("\n⏳ This may take 2-3 minutes. Please wait...\n")

try:
    # Execute the workflow (async)
    result = await run_odd_workflow(
        scenario_name=SCENARIO_NAME,
        nl_odd_description=nl_odd_description
    )
    
    print("✅ Workflow completed successfully!")
    print(f"\n📊 Analysis Results Summary:")
    print(f"   • Windows analyzed: {result['report']['scenario_metadata']['total_windows_analyzed']}")
    print(f"   • Data source: {result['report']['scenario_metadata']['data_source']}")
    print(f"   • ODD compliance: {result['full_analysis']['odd_compliance']['overall_compliance']}")
    
    # Store result for visualization
    workflow_result = result
    
except Exception as e:
    print(f"❌ Workflow failed: {e}")
    import traceback
    traceback.print_exc()
    workflow_result = None

In [ ]:
if workflow_result:
    report = workflow_result['report']
    
    print("=" * 80)
    print("EXECUTIVE SUMMARY")
    print("=" * 80)
    print(report['executive_summary'])
    print()
    
    print("=" * 80)
    print("KEY FINDINGS")
    print("=" * 80)
    for i, finding in enumerate(report['key_findings'], 1):
        print(f"{i}. {finding}")
    print()
    
    print("=" * 80)
    print("RECOMMENDATIONS")
    print("=" * 80)
    for i, rec in enumerate(report['recommendations'], 1):
        print(f"{i}. {rec}")
else:
    print("⚠️ No results available. Please run the workflow first.")

In [ ]:
if workflow_result:
    compliance = workflow_result['full_analysis']['odd_compliance']
    
    print("🎯 ODD COMPLIANCE ANALYSIS")
    print("=" * 80)
    print(f"\n Overall Status: {compliance['overall_compliance']}")
    print(f" Violations: {len(compliance.get('violations', []))}")
    print()
    
    if compliance.get('violations'):
        print(" ❌ VIOLATIONS DETECTED:")
        for violation in compliance['violations']:
            print(f"    • {violation}")
    else:
        print(" ✅ No violations detected")
    
    print("\n📊 CATEGORICAL COMPLIANCE:")
    print("-" * 80)
    for axis, status in compliance.get('categorical_compliance', {}).items():
        icon = "✅" if status == "IN_ODD" else ("⚠️" if status == "ODD_BOUNDARY" else "❌")
        print(f"{icon} {axis:30s} {status}")
    
    print("\n📏 NUMERIC COMPLIANCE:")
    print("-" * 80)
    for axis, status in compliance.get('numeric_compliance', {}).items():
        icon = "✅" if status == "IN_ODD" else ("⚠️" if status == "ODD_BOUNDARY" else "❌")
        print(f"{icon} {axis:30s} {status}")
else:
    print("⚠️ No results available.")

In [ ]:
if workflow_result:
    collision_data = workflow_result['full_analysis']['collision']
    per_window = collision_data.get('per_window_collision', [])
    
    if per_window:
        # Extract data for plotting
        windows = [w['window_id'] for w in per_window]
        risk_levels = [w['risk_level'] for w in per_window]
        likelihoods = [w['likelihood'] for w in per_window]
        
        # Create color map
        color_map = {'safe': 'green', 'caution': 'orange', 'alert': 'red'}
        colors = [color_map.get(level, 'gray') for level in risk_levels]
        
        # Create visualization
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
        
        # Plot 1: Risk level bars
        ax1.bar(range(len(windows)), 
                [1 if r == 'safe' else 2 if r == 'caution' else 3 for r in risk_levels],
                color=colors, alpha=0.7, edgecolor='black')
        ax1.set_ylabel('Risk Level', fontsize=12, fontweight='bold')
        ax1.set_yticks([1, 2, 3])
        ax1.set_yticklabels(['Safe', 'Caution', 'Alert'])
        ax1.set_title('Collision Risk Levels per Window', fontsize=14, fontweight='bold')
        ax1.grid(axis='y', alpha=0.3)
        
        # Plot 2: Likelihood scores
        ax2.plot(range(len(windows)), likelihoods, 'o-', linewidth=2, markersize=8, 
                 color='darkblue', label='Collision Likelihood')
        ax2.axhline(y=0.3, color='orange', linestyle='--', label='Boundary (0.3)')
        ax2.axhline(y=0.5, color='red', linestyle='--', label='Alert Threshold (0.5)')
        ax2.fill_between(range(len(windows)), 0, 0.3, alpha=0.1, color='green')
        ax2.fill_between(range(len(windows)), 0.3, 0.5, alpha=0.1, color='orange')
        ax2.fill_between(range(len(windows)), 0.5, 1.0, alpha=0.1, color='red')
        ax2.set_xlabel('Window ID', fontsize=12, fontweight='bold')
        ax2.set_ylabel('Likelihood Score', fontsize=12, fontweight='bold')
        ax2.set_title('Collision Likelihood Scores', fontsize=14, fontweight='bold')
        ax2.set_xticks(range(len(windows)))
        ax2.set_xticklabels(windows)
        ax2.legend()
        ax2.grid(alpha=0.3)
        ax2.set_ylim(0, 1.0)
        
        plt.tight_layout()
        plt.show()
        
        # Print statistics
        print("\n📊 COLLISION RISK STATISTICS:")
        stats = collision_data.get('overall_stats', {})
        print(f"   Total windows: {stats.get('total_windows', 0)}")
        print(f"   Alert level: {stats.get('alert_count', 0)} ({stats.get('alert_count', 0)/len(windows)*100:.1f}%)")
        print(f"   Caution level: {stats.get('caution_count', 0)} ({stats.get('caution_count', 0)/len(windows)*100:.1f}%)")
        print(f"   Safe: {stats.get('safe_count', 0)} ({stats.get('safe_count', 0)/len(windows)*100:.1f}%)")
        print(f"   Avg likelihood: {stats.get('avg_likelihood', 0):.3f}")
    else:
        print("⚠️ No per-window collision data available")
else:
    print("⚠️ No results available.")

In [ ]:
if workflow_result:
    motion_data = workflow_result['full_analysis']['motion']
    stats = motion_data.get('motion_stats', {})
    
    print("🎯 IMU-BASED MOTION DETECTION")
    print("=" * 80)
    print(f"Motion detected: {stats.get('motion_detected_count', 0)}/{len(motion_data.get('windows_analyzed', []))} windows")
    print(f"Detection rate: {stats.get('motion_detection_rate', 0)*100:.1f}%")
    print(f"Overall assessment: {stats.get('overall_assessment', 'N/A')}")
    print()
    
    print("📊 MOTION CHARACTERISTICS:")
    print("-" * 80)
    print(f"Max horizontal accel: {stats.get('max_horizontal_accel_mps2', 0):.3f} m/s²")
    print(f"Max angular velocity: {stats.get('max_angular_velocity_radps', 0):.3f} rad/s")
    print(f"Max platform tilt: {stats.get('max_platform_tilt_deg', 0):.1f}°")
    print()
    
    # Motion type distribution
    motion_types = stats.get('motion_type_distribution', {})
    if motion_types:
        print("🔄 MOTION TYPE DISTRIBUTION:")
        print("-" * 80)
        for motion_type, count in motion_types.items():
            print(f"{motion_type:15s}: {count} windows")
    
    print()
    print("💡 Note: Motion detection uses IMU accelerometer (>0.05 m/s²) and")
    print("   gyroscope (>0.1 rad/s) instead of odometry (which was broken).")
else:
    print("⚠️ No results available.")

In [ ]:
if workflow_result:
    # Save to JSON
    output_path = scenario_path / "odd_analysis_report_notebook.json"
    with open(output_path, 'w') as f:
        json.dump(workflow_result, f, indent=2)
    
    print(f"✅ Results exported to: {output_path}")
    print(f"   File size: {output_path.stat().st_size / 1024:.1f} KB")
    
    # Also save a summary text file
    summary_path = scenario_path / "odd_analysis_summary.txt"
    with open(summary_path, 'w') as f:
        f.write("=" * 80 + "\n")
        f.write("ODD/COD ANALYSIS SUMMARY\n")
        f.write("=" * 80 + "\n\n")
        f.write(workflow_result['report']['executive_summary'] + "\n\n")
        f.write("KEY FINDINGS:\n")
        for i, finding in enumerate(workflow_result['report']['key_findings'], 1):
            f.write(f"{i}. {finding}\n")
        f.write("\nRECOMMENDATIONS:\n")
        for i, rec in enumerate(workflow_result['report']['recommendations'], 1):
            f.write(f"{i}. {rec}\n")
    
    print(f"✅ Summary exported to: {summary_path}")
else:
    print("⚠️ No results to export.")

## 10. Next Steps and Customization

Now that you've run the complete workflow, here are some ways to customize and extend it:

### 🎯 Customize Your ODD

Modify the `nl_odd_description` in cell 3 to match your specific robot and deployment:
- **Outdoor delivery robot**: Add weather, GPS, terrain slope constraints
- **Aerial drone**: Add altitude, wind speed, battery, GPS quality constraints  
- **Warehouse AMR**: Add floor type, shelf proximity, human traffic constraints

### 📊 Test Different Scenarios

Change `SCENARIO_NAME` in cell 4 to analyze different datasets:
- Real robot data (from physical deployment)
- Simulation data (various environment types)
- Edge cases (boundary conditions, failure modes)

### 🔬 Advanced Analysis

- **Compare multiple scenarios**: Run workflow on different datasets and compare compliance
- **Parameter sensitivity**: Test how ODD threshold changes affect compliance
- **Time-series analysis**: Track ODD compliance over long-term deployments
- **Failure mode analysis**: Identify which ODD violations are most common

### 🚀 Production Deployment

To use this in production:
1. Preprocess ROS2 bags with `extract_windows.py`
2. Run workflow on each scenario
3. Aggregate results for fleet-wide monitoring
4. Set up alerts for ODD violations

### 📚 Learn More

- **Documentation**: `docs/guides/GETTING_STARTED.md`
- **Script source**: `scripts/odd_workflow_full.py`
- **Example reports**: `docs/examples/`
- **Project updates**: `IMU_MOTION_DETECTION_UPDATE.md`

## 9. Export Results

Save the complete analysis report to JSON for further processing or sharing.

## 8. Motion Detection Analysis

View IMU-based motion detection statistics (using accelerometer/gyroscope data).

## 7. Visualize Collision Risk Timeline

Track collision risk levels across all analyzed windows.

## 6. ODD Compliance Analysis

View the detailed ODD vs COD comparison and violation detection.

## 5. View Executive Summary

Let's start with the high-level findings from the analysis.

## 4. Run the 10-Agent Workflow

Now we'll execute the complete ODD/COD analysis workflow. This will:

1. **ODD Spec Agent**: Convert your NL description to formal specification
2. **Perception Agents**: Analyze camera + LiDAR images (loop + summary)
3. **Motion Agents**: Detect motion from IMU data (loop + summary)
4. **Collision Agents**: Assess collision risk (loop + summary)
5. **COD Classifier**: Determine current operating domain
6. **ODD Compliance**: Compare COD vs ODD, detect violations
7. **Report Generator**: Create comprehensive analysis report

This typically takes 2-3 minutes for a 13-window scenario.

## 3. Select Scenario to Analyze

Choose which preprocessed scenario you want to analyze. The data should be in `data/processed/runs/` directory.

## 2. Define Your ODD Specification

Describe your robot's operational constraints in natural language. The **ODD Spec Agent** will convert this to a formal specification.

## 1. Setup and Configuration

First, let's configure the Google Gemini API and import required libraries.

# Go2 ODD/COD Observer - Complete 10-Agent Workflow

This notebook demonstrates the complete **ODD-first** workflow for analyzing Operational Design Domain (ODD) compliance for Unitree Go2 robot scenarios.

**Key Concepts:**
- **ODD (Operational Design Domain)**: The environment the robot is **designed** to work in (specification)
- **COD (Current Operating Domain)**: The environment the robot is **actually** in (measured from sensors)
- **ODD Compliance**: Comparison of COD against ODD to detect violations

**10-Agent Sequential Pipeline:**
1. **ODD Spec Agent** - Convert natural language to formal ODD specification (runs FIRST)
2. **Perception Loop + Summary** - Analyze camera + LiDAR images, classify sim vs real
3. **Motion Loop + Summary** - IMU-based motion detection (accelerometer/gyroscope)
4. **Collision Loop + Summary** - Multimodal fusion for risk assessment
5. **COD Classifier** - Classify current operating domain from sensor data
6. **ODD Compliance** - Compare COD vs ODD, detect violations
7. **Report Generator** - Create comprehensive human-readable report

**Note:** This workflow assumes you have preprocessed ROS2 bag files into time-windowed snapshots using the `extract_windows.py` script.

## 1. Setup and Dependencies

Install and import required packages for Google AI SDK and our analysis framework.

In [ ]:
# ============================================================================
# DATA ACCESS TOOLS (agents call these to get scenario data)
# Tools use DIRECT FILE READING - no preloading required
# ============================================================================

import base64

def get_scenario_data(scenario_path: str) -> dict:
    """
    Retrieve scenario data including motion JSON for each window by reading files directly.
    
    AGENTS MUST call this tool first to see what windows are available.
    Do NOT make up window IDs - use only what this tool returns.
    
    Args:
        scenario_path: str - Path to the scenario directory
    
    Returns:
        dict with keys:
            - "status": "success" or "error"
            - "scenario_name": Name of the scenario
            - "total_windows": Number of windows
            - "windows": List of dicts with window_id and motion_json
    """
    try:
        from pathlib import Path
        import json
        import pandas as pd
        
        scenario_path = Path(scenario_path)
        if not scenario_path.exists():
            return {
                "status": "error",
                "error_message": f"Scenario path not found: {scenario_path}"
            }
        
        # Find and load index CSV
        index_files = list(scenario_path.glob("index_*.csv"))
        if not index_files:
            return {
                "status": "error",
                "error_message": f"No index file found in {scenario_path}"
            }
        
        index_df = pd.read_csv(index_files[0])
        scenario_name = scenario_path.name
        
        windows = []
        for _, row in index_df.iterrows():
            window_id = str(row['window_id']).zfill(3)
            motion_file = scenario_path / f"motion_{scenario_name}_w{window_id}.json"
            
            if motion_file.exists():
                with open(motion_file, 'r') as f:
                    motion_json = json.load(f)
                
                windows.append({
                    "window_id": window_id,
                    "motion_json": motion_json
                })
        
        return {
            "status": "success",
            "scenario_name": scenario_name,
            "total_windows": len(windows),
            "windows": windows
        }
    except Exception as e:
        return {
            "status": "error",
            "error_message": f"Failed to get scenario data: {str(e)}"
        }


def get_window_image_raw(window_id: str, image_type: str, scenario_path: str) -> dict:
    """
    Retrieve a window image as base64-encoded PNG (efficient for large images).
    This encoding can be processed directly by Gemini's vision API.
    
    Args:
        window_id: str - Window identifier (e.g., "006")
        image_type: str - "camera", "bev_occupancy", "bev_height", "bev_density", "bev_roughness"
        scenario_path: str - Path to the scenario directory
    
    Returns:
        dict with "success", "image_base64", "mime_type", "size_kb", "format"
        (returns base64 for efficient serialization in agent messages)
    """
    try:
        from pathlib import Path
        
        scenario_path = Path(scenario_path)
        scenario_name = scenario_path.name
        
        # Construct filename based on image type
        if image_type == "camera":
            filename = f"cam_{scenario_name}_w{window_id}.png"
        elif image_type.startswith("bev_"):
            channel = image_type.replace("bev_", "")
            filename = f"bev_{channel}_{scenario_name}_w{window_id}.png"
        else:
            return {
                "status": "error",
                "error_message": f"Unknown image type: {image_type}"
            }
        
        file_path = scenario_path / filename
        if not file_path.exists():
            return {
                "status": "error",
                "error_message": f"Image not found: {filename}"
            }
        
        with open(file_path, 'rb') as f:
            image_bytes = f.read()
        
        # Return as base64 to avoid binary serialization issues
        image_base64 = base64.b64encode(image_bytes).decode('utf-8')
        
        return {
            "status": "success",
            "image_base64": image_base64,
            "mime_type": "image/png",
            "size_kb": len(image_bytes) / 1024,
            "format": ".png",
            "encoding": "base64"
        }
    except Exception as e:
        return {
            "status": "error",
            "error_message": f"Failed to retrieve image: {str(e)}"
        }

def get_window_image_base64(window_id: str, image_type: str, scenario_path: str) -> dict:
    """
    Retrieve a window image as base64-encoded PNG data.
    
    Args:
        window_id: str - Window identifier (e.g., "006")
        image_type: str - "camera", "bev_occupancy", "bev_height", "bev_density", "bev_roughness"
        scenario_path: str - Path to the scenario directory
    
    Returns:
        dict with "success", "image_base64", "mime_type", "size_kb", "format"
    """
    try:
        from pathlib import Path
        
        scenario_path = Path(scenario_path)
        scenario_name = scenario_path.name
        
        # Construct filename based on image type
        if image_type == "camera":
            filename = f"cam_{scenario_name}_w{window_id}.png"
        elif image_type.startswith("bev_"):
            channel = image_type.replace("bev_", "")
            filename = f"bev_{channel}_{scenario_name}_w{window_id}.png"
        else:
            return {
                "status": "error",
                "error_message": f"Unknown image type: {image_type}"
            }
        
        file_path = scenario_path / filename
        if not file_path.exists():
            return {
                "status": "error",
                "error_message": f"Image not found: {filename}"
            }
        
        with open(file_path, 'rb') as f:
            image_bytes = f.read()
        
        # Encode to base64
        image_base64 = base64.b64encode(image_bytes).decode('utf-8')
        
        return {
            "status": "success",
            "image_base64": image_base64,
            "mime_type": "image/png",
            "size_kb": len(image_bytes) / 1024,
            "format": ".png",
            "encoding": "base64"
        }
    except Exception as e:
        return {
            "status": "error",
            "error_message": f"Failed to retrieve image: {str(e)}"
        }


print("✓ Direct file-reading tools defined (no preloading required)")


In [ ]:
# ============================================================================
# VISUALIZATION TOOLS (for Report Agent)
# ============================================================================

from typing import List

def generate_distance_plot(times: List[float], distances: List[float], title: str = "ODD Distance over Time") -> dict:
    """
    Generate a timeline plot showing how close the robot is to violating the ODD.
    
    Args:
        times: List of time values (seconds)
        distances: List of distance metrics (0=boundary, 1=fully compliant)
        title: Plot title
    
    Returns:
        dict with "success", "plot_base64", or "error_message"
    """
    try:
        import matplotlib.pyplot as plt
        import base64
        from io import BytesIO
        
        fig, ax = plt.subplots(figsize=(12, 6))
        ax.plot(times, distances, linewidth=2, marker='o', markersize=4)
        ax.axhline(y=0, color='r', linestyle='--', label='ODD Boundary')
        ax.axhline(y=1, color='g', linestyle='--', label='Fully Compliant')
        ax.set_xlabel('Time (s)')
        ax.set_ylabel('Distance from ODD')
        ax.set_title(title)
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        buf = BytesIO()
        fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
        buf.seek(0)
        plot_bytes = buf.read()
        plt.close(fig)
        
        plot_base64 = base64.b64encode(plot_bytes).decode('utf-8')
        
        return {
            "status": "success",
            "plot_base64": plot_base64,
            "format": "png",
            "title": title
        }
    except Exception as e:
        return {
            "status": "error",
            "error_message": f"Failed to generate plot: {str(e)}"
        }


def generate_status_distribution(statuses: List[str], title: str = "ODD Compliance Status Distribution") -> dict:
    """
    Generate a bar chart showing distribution of ODD compliance statuses.
    
    Args:
        statuses: List of status strings ("in_odd", "near_boundary", "odd_exit")
        title: Plot title
    
    Returns:
        dict with "success", "plot_base64", or "error_message"
    """
    try:
        import matplotlib.pyplot as plt
        import base64
        from io import BytesIO
        from collections import Counter
        
        counts = Counter(statuses)
        labels = list(counts.keys())
        values = list(counts.values())
        
        colors = {
            "in_odd": "green",
            "near_boundary": "orange",
            "odd_exit": "red"
        }
        bar_colors = [colors.get(label, "blue") for label in labels]
        
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.bar(labels, values, color=bar_colors, alpha=0.7)
        ax.set_ylabel('Count')
        ax.set_title(title)
        ax.grid(True, alpha=0.3, axis='y')
        
        # Add value labels on bars
        for i, v in enumerate(values):
            ax.text(i, v + 0.5, str(v), ha='center', va='bottom')
        
        buf = BytesIO()
        fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
        buf.seek(0)
        plot_bytes = buf.read()
        plt.close(fig)
        
        plot_base64 = base64.b64encode(plot_bytes).decode('utf-8')
        
        return {
            "status": "success",
            "plot_base64": plot_base64,
            "format": "png",
            "title": title,
            "distribution": dict(counts)
        }
    except Exception as e:
        return {
            "status": "error",
            "error_message": f"Failed to generate status distribution: {str(e)}"
        }


print("✓ Visualization tools defined")


In [ ]:
# ============================================================================
# DEFINE SCENARIO PATH AND CREATE FUNCTION TOOLS
# ============================================================================

from pathlib import Path

# Define dataset path
PROJECT_ROOT = Path("/workspaces/go2-odd-observer")
DATA_DIR = PROJECT_ROOT / "data" / "processed" / "runs"
scenario_path = DATA_DIR / "sim_run_test"  # Change this to analyze different datasets

print(f"Dataset path: {scenario_path}")
if scenario_path.exists():
    print(f"✓ Dataset found")
    files = list(scenario_path.glob("*"))
    print(f"  Files: {len(files)}")
else:
    print(f"✗ Dataset NOT found!")

# Create FunctionTool wrappers for agents to call
from google.adk.tools import FunctionTool

# Create tools that pass scenario_path to the functions
def scenario_data_wrapper() -> dict:
    """Wrapper that includes scenario_path."""
    return get_scenario_data(str(scenario_path))

def image_raw_wrapper(window_id: str, image_type: str) -> dict:
    """Wrapper that includes scenario_path for raw image reading."""
    return get_window_image_raw(window_id, image_type, str(scenario_path))

def image_base64_wrapper(window_id: str, image_type: str) -> dict:
    """Wrapper that includes scenario_path for base64 image reading."""
    return get_window_image_base64(window_id, image_type, str(scenario_path))

# Register as FunctionTools
scenario_data_tool = FunctionTool(func=scenario_data_wrapper)
get_image_tool_raw = FunctionTool(func=image_raw_wrapper)
get_image_tool_base64 = FunctionTool(func=image_base64_wrapper)
distance_plot_tool = FunctionTool(func=generate_distance_plot)
status_dist_tool = FunctionTool(func=generate_status_distribution)

# Use raw bytes by default (more efficient)
get_image_tool = get_image_tool_raw

print("✓ FunctionTools created for data and visualization access")
print("  - scenario_data_tool: Get scenario metadata and motion JSON")
print("  - get_image_tool: Get images as raw PNG bytes (efficient, default)")
print("  - get_image_tool_base64: Get images as base64 (compatibility)")
print("  - distance_plot_tool: Generate ODD distance timeline plots")
print("  - status_dist_tool: Generate status distribution charts")


In [ ]:
# Install Google Agent Development Kit (ADK) and dependencies
# Note: Run this cell only once or when packages need updating
!pip install -q google-adk python-dotenv

In [ ]:
import sys
from pathlib import Path

# Add project root to Python path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Standard library imports
import json
import base64
import io
from typing import Dict, List, Tuple, Any

# Third-party imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# Google ADK imports
from google.genai import types
from google.adk.agents import Agent, SequentialAgent, ParallelAgent, LoopAgent
from google.adk.models.google_llm import Gemini
from google.adk.tools import FunctionTool
from google.adk.runners import InMemoryRunner
from google.adk.tools import google_search, AgentTool, ToolContext

## 2. Configuration (REQUIRED)

### 2.1 Google Gemini API Key

**This notebook requires a Google Gemini API key** to demonstrate AI agents in action.

Get your free API key at: https://aistudio.google.com/app/apikey

### 2.2 Model Selection Strategy

**Cost-Optimized Multi-Model Approach:**
- `gemini-2.5-pro`: Vision analysis, data aggregation, multimodal fusion (accuracy critical)
- `gemini-2.0-flash-lite`: ODD spec, COD classification, compliance (simple synthesis)
- **Result:** ~30% cost savings while maintaining quality

In [ ]:
# ============================================
# 2.1 Configure Google API Key
# ============================================
import os
from dotenv import load_dotenv

# Option 1: Set via environment variable (RECOMMENDED)
# export GOOGLE_API_KEY='your-api-key-here'

# Option 2: Load from .env file
load_dotenv()

# Option 3: Set directly in notebook (NOT recommended - avoid committing keys!)
# os.environ['GOOGLE_API_KEY'] = 'your-api-key-here'

# Verify API key is configured
GOOGLE_API_KEY = os.getenv('GOOGLE_API_KEY')
if GOOGLE_API_KEY:
    print("✓ Google AI SDK configured successfully")
    print("  API key detected")
else:
    print("❌ GOOGLE_API_KEY not found!")
    print()
    print("This notebook REQUIRES a Google Gemini API key to run.")
    print()
    print("To get a free API key:")
    print("  1. Visit https://aistudio.google.com/app/apikey")
    print("  2. Create or select a project")
    print("  3. Generate an API key")
    print()
    print("  To configure your API key:")
    print("  export GOOGLE_API_KEY='your-key-here'")
    print("  OR create a .env file with: GOOGLE_API_KEY=your-key-here")
    print()
    print("⚠ The notebook will FAIL without an API key - this is intentional!")
    print("  Falling back to fake data would defeat the purpose of learning about AI agents.")

# ============================================
# 2.2 Multi-Model Configuration (Cost Optimized)
# ============================================

# Vision/aggregation: High accuracy model
GEMINI_MODEL_PRO = "gemini-2.5-pro"

# Simple synthesis: Cost-efficient model  
GEMINI_MODEL_LITE = "gemini-2.0-flash-lite"

retry_config = types.HttpRetryOptions(
    attempts=5,
    exp_base=7,
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504],
)

print(f"✓ Pro model (vision/fusion): {GEMINI_MODEL_PRO}")
print(f"✓ Lite model (synthesis): {GEMINI_MODEL_LITE}")
print("✓ Agent config: JSON response format, temperature=0.1")

## 3. User Inputs

Define what you want to analyze:
1. **Natural language ODD**: Operating constraints in plain English (provided to ODD Spec Agent)
2. **Dataset path**: Location of preprocessed window data

The 10-agent workflow (Section 5+) will:
- Convert NL ODD → formal specification
- Analyze sensors → measure COD
- Compare COD vs ODD → detect violations

## 3.5 Direct File Reading Tools

Agents access scenario data and images directly through tools. No pre-loading required.
Tools read files on-demand and return data in both raw bytes (efficient) and base64 formats.


In [ ]:
# Natural language ODD definition for Unitree Go2 indoor navigation

odd_natural_language = """
The Unitree Go2 quadruped robot is designed for indoor navigation in office environments.

OPERATIONAL CONSTRAINTS:

1. Environment Type:
   - Designed for: indoor_office, indoor_corridor
   - Prohibited: outdoor environments, staircases, unstructured terrain

2. Lighting Conditions:
   - Required: bright or dim lighting (adequate visibility)
   - Prohibited: dark environments (requires vision sensors)

3. Terrain Type:
   - Designed for: smooth_floor (tile, hardwood, low-pile carpet)
   - Prohibited: rough or very_rough terrain, stairs, slopes >10°

4. Speed Range:
   - Normal operation: 0.0 to 1.5 m/s
   - Physical limit: 2.5 m/s (emergency only)

5. Obstacle Density:
   - Acceptable: low to moderate obstacles (0.0 to 0.6 normalized)
   - Boundary: 0.6 to 0.8 (crowded but navigable)
   - Prohibited: >0.8 (too cluttered for safe navigation)

6. Traversability:
   - Required: navigable space (0.5 to 1.0 score)
   - Boundary: 0.3 to 0.5 (challenging but possible)
   - Prohibited: <0.3 (impassable or unsafe)

7. Collision Risk:
   - Acceptable: low risk (0.0 to 0.3 likelihood)
   - Boundary: 0.3 to 0.5 (caution required)
   - Prohibited: >0.5 (high risk, stop immediately)

8. Platform Stability:
   - Required: stable platform (roll/pitch <15°)
   - Boundary: 15° to 20° (unstable but recoverable)
   - Prohibited: >20° (tip-over risk)
"""

# Dataset to analyze
SCENARIO_NAME = "sim_run_new"  # Change to your scenario
SCENARIO_PATH = f"../data/processed/runs/{SCENARIO_NAME}"

print(f"✓ ODD specification: {len(odd_natural_language)} characters")
print(f"✓ Scenario: {SCENARIO_NAME}")
print(f"✓ Path: {SCENARIO_PATH}")

## 4. Define Tool Functions for Agents

These Python functions will be available as tools for the orchestrator agent to call.
They provide access to: file I/O, ODD spec construction, COD computation, and visualization.

## 5. Define Specialist Agents

Create individual agents using the Google ADK (Agent Development Kit) following the Kaggle Day 1B pattern.
Each agent is a specialist that performs one specific analysis task.

### 5.0 Data Loader Agent

Data is pre-loaded locally and passed to agents via session state.
No Data Loader Agent needed - cleaner architecture focused on reasoning, not I/O.

In [ ]:
# ODD Spec Agent: Converts natural language ODD → structured JSON
odd_spec_agent = Agent(
    name="ODD_Spec_Parser",
    model=Gemini(
        model=GEMINI_MODEL,
        api_key=GOOGLE_API_KEY
    ),
    instruction="""You are an expert in robotic operational design domains (ODD).

You will receive a natural language ODD specification in the invocation context.
Convert it to structured JSON that defines the robot's operational boundaries.

Output valid JSON only with this schema:
{
  "version": "1.0",
  "description": "<brief summary>",
  "axes": {
    "speed": {
      "type": "numeric",
      "feature": "avg_forward_speed",
      "units": "m/s",
      "in_odd": [min, max],
      "near_boundary": [min, max],
      "hard_limit": [min, max]
    },
    "roll_pitch": {
      "type": "numeric",
      "feature": "max_abs_roll_pitch_deg",
      "units": "degrees",
      "in_odd": [min, max],
      "near_boundary": [min, max],
      "hard_limit": [min, max]
    },
    "terrain": {
      "type": "categorical",
      "feature": "terrain_roughness_class",
      "allowed_in_odd": ["smooth", "moderate"],
      "allowed_all": ["smooth", "moderate", "rough", "very_rough"]
    },
    "lighting": {
      "type": "categorical",
      "feature": "lighting_class",
      "allowed_in_odd": ["bright", "dim"],
      "allowed_all": ["bright", "dim", "dark"]
    },
    "humans_close": {
      "type": "categorical",
      "feature": "humans_very_close",
      "allowed_in_odd": [false],
      "allowed_all": [true, false]
    },
    "collision": {
      "type": "categorical",
      "feature": "collision_suspected",
      "allowed_in_odd": [false],
      "allowed_all": [true, false]
    }
  },
  "importance": {
    "speed": 1.0,
    "roll_pitch": 1.2,
    "terrain": 1.0,
    "lighting": 0.8,
    "humans_close": 1.5,
    "collision": 2.0
  }
}

Extract ranges, categorical values, and importance weights from the input.""",
    output_key="odd_spec_json"
)

print("✅ ODD Spec Agent created")

### 5.2 Motion Analysis Agent

Extracts motion features from velocity and IMU time series data.

In [ ]:
# Motion Analysis Agent: Extracts motion features from sensor data
motion_agent = Agent(
    name="Motion_Analyzer",
    model=Gemini(
        model=GEMINI_MODEL,
        api_key=GOOGLE_API_KEY
    ),
    tools=[scenario_data_tool],
    instruction="""You are a motion analysis expert for mobile robots.

CRITICAL INSTRUCTIONS:
1. FIRST: Call get_scenario_data() tool to retrieve actual window IDs and motion JSON
2. Do NOT make up window IDs or data - use ONLY what the tool returns
3. For EACH window returned by the tool, extract motion features
4. Return results for ALL windows from the tool

From the shared state, you also have access to:
- odd_spec_json: The formal ODD specification with motion feature constraints

For each window in the scenario data, analyze the motion JSON and extract features:

Output valid JSON with this schema:
{
  "windows": [
    {
      "window_id": "006",
      "avg_forward_speed": <float m/s>,
      "max_forward_speed": <float m/s>,
      "max_abs_roll_pitch_deg": <float degrees>,
      "motion_label": "smooth" | "dynamic"
    },
    ...for each window...
  ]
}

IMPORTANT:
- Process ALL windows from get_scenario_data()
- Extract real motion metrics from the motion_json for each window
- Do not skip windows or return fewer windows than provided by the tool
- Return complete analysis for every window""",
    output_key="motion_features"
)

print("✅ Motion Agent created")

### 5.3 Vision Analysis Agent

Classifies environmental conditions from camera images.

In [ ]:
# Vision Analysis Agent: Classifies environmental conditions from images
vision_agent = Agent(
    name="Vision_Analyzer",
    model=Gemini(
        model=GEMINI_MODEL,
        api_key=GOOGLE_API_KEY
    ),
    tools=[scenario_data_tool, get_image_tool],
    instruction="""You are a computer vision expert for mobile robots.

CRITICAL INSTRUCTIONS:
1. FIRST: Call get_scenario_data() tool to get actual window IDs
2. Do NOT make up window IDs - use ONLY what the tool returns
3. For EACH window from the tool, call get_window_image("camera") to retrieve the camera image
4. ANALYZE the image immediately when you receive it
5. DO NOT include the raw image bytes in your JSON output - just the analysis results
6. Return results for ALL windows

From the shared state, you also have access to:
- odd_spec_json: Environmental constraints (lighting, human proximity)

For each window, analyze the CAMERA IMAGE you retrieved and extract:
- lighting_class: "bright" | "dim" | "low_light"
- humans_visible: true | false
- humans_very_close: true | false (within 1 meter)
- environment_type: "office" | "hallway" | "stairwell" | "outdoor" | "other"
- detected_hazards: list of objects/conditions that could affect deployment

Output valid JSON with this schema:
{
  "windows": [
    {
      "window_id": "006",
      "lighting_class": "bright",
      "humans_visible": false,
      "humans_very_close": false,
      "environment_type": "office",
      "detected_hazards": []
    },
    ...for each window...
  ]
}
      "detected_hazards": []
    },
    ...for each window...
  ]
}

IMPORTANT:
- Process ALL windows from get_scenario_data()
- Analyze the ACTUAL IMAGES retrieved via get_window_image() tool
- Extract features from images but do NOT include image bytes in output
- Return complete analysis for every window""",
    output_key="vision_features"
)

print("✅ Vision Agent created")


### 5.4 Terrain Analysis Agent

Analyzes LiDAR Bird's Eye View images to classify terrain roughness.

In [ ]:
# Terrain Analysis Agent: Evaluates terrain roughness and traversability from BEV images
terrain_agent = Agent(
    name="Terrain_Analyzer",
    model=Gemini(
        model=GEMINI_MODEL,
        api_key=GOOGLE_API_KEY
    ),
    tools=[scenario_data_tool, get_image_tool],
    instruction="""You are a terrain analysis expert for mobile robots. You specialize in analyzing LiDAR Bird's Eye View (BEV) maps.

CRITICAL INSTRUCTIONS:
1. FIRST: Call get_scenario_data() tool to get actual window IDs
2. Do NOT make up window IDs - use ONLY what the tool returns
3. For EACH window from the tool, retrieve BEV images using get_window_image()
4. Call get_window_image() with image_type: "bev_occupancy", "bev_height", "bev_density", "bev_roughness"
5. ANALYZE each retrieved BEV image immediately
6. DO NOT include the raw image bytes in your JSON output - just the analysis results
7. Return results for ALL windows

From the shared state, you also have access to:
- odd_spec_json: Terrain constraints (smooth, moderate, rough, very_rough)

For each window, analyze the BEV MAPS you retrieved and assess:
- Occupancy map: percentage of grid occupied by obstacles
- Height map: typical obstacle heights
- Roughness map: surface roughness and irregularities
- Density map: point cloud density indicating surface coverage

Output valid JSON with this schema:
{
  "windows": [
    {
      "window_id": "006",
      "terrain_roughness_class": "smooth" | "moderate" | "rough" | "very_rough",
      "occupancy_ratio": <float 0-1>,
      "obstacle_density": <float 0-1>,
      "traversability_score": <float 0-1>,
      "hazard_regions": []
    },
    ...for each window...
  ]
}

IMPORTANT:
- Process ALL windows from get_scenario_data()
- Retrieve ALL four BEV images (occupancy, height, density, roughness) for each window
- Analyze the BEV maps but DO NOT include image bytes in output
- Look for sparse areas (good), dense areas (obstacles), and irregular patterns (roughness)
- Return complete analysis for every window""",
    output_key="terrain_features"
)

print("✅ Terrain Agent created")


### 5.5 Collision Detection Agent

Performs multi-modal sensor fusion to detect collision events.

In [ ]:
# Collision Detection Agent: Identifies collision risks from sensor data and images
collision_agent = Agent(
    name="Collision_Detector",
    model=Gemini(
        model=GEMINI_MODEL,
        api_key=GOOGLE_API_KEY
    ),
    tools=[scenario_data_tool, get_image_tool],
    instruction="""You are a collision detection and obstacle avoidance expert for mobile robots.

CRITICAL INSTRUCTIONS:
1. FIRST: Call get_scenario_data() tool to get actual window IDs
2. Do NOT make up window IDs - use ONLY what the tool returns
3. For EACH window from the tool, retrieve BOTH camera and BEV images
4. Call get_window_image() with: "camera", "bev_occupancy", "bev_height"
5. ANALYZE each retrieved image immediately
6. DO NOT include the raw image bytes in your JSON output - just the analysis results
7. Return results for ALL windows

From the shared state, you also have access to:
- odd_spec_json: Collision policy (zero collisions tolerated)

For each window, analyze the IMAGES you retrieved to detect collision risks:
- Camera image: Look for obstacles, walls, objects directly in front
- BEV occupancy: Analyze occupancy grid to detect obstacles around the robot
- BEV height map: Check for obstacles of various heights

Output valid JSON with this schema:
{
  "windows": [
    {
      "window_id": "006",
      "collision_suspected": true | false,
      "collision_confidence": <float 0-1>,
      "collision_type": "none" | "obstacle" | "wall" | "human" | "unknown",
      "risk_level": "safe" | "warning" | "danger",
      "notes": "Description of what collision hazard was detected, if any"
    },
    ...for each window...
  ]
}

IMPORTANT:
- Process ALL windows from get_scenario_data()
- Retrieve MULTIPLE image types for each window (camera + BEV)
- Analyze the retrieved images but DO NOT include image bytes in output
- Look for obstacles in both front-facing (camera) and surrounding (BEV) views
- Return complete analysis for every window""",
    output_key="collision_features"
)

print("✅ Collision Agent created")


### 5.6 COD Evaluator Agent

Specialist agent that coordinates COD computation using mathematical tool functions.

In [ ]:
# COD Evaluator Agent: Aggregates sensor analysis results against ODD
cod_evaluator_agent = Agent(
    name="COD_Evaluator",
    model=Gemini(
        model=GEMINI_MODEL,
        api_key=GOOGLE_API_KEY
    ),
    instruction="""You are a Conditions of Deployment (COD) evaluation expert.

From the shared state, you have access to:
- odd_spec_json: The formal ODD specification
- motion_features: Motion analysis for all windows
- vision_features: Vision analysis for all windows
- terrain_features: Terrain analysis for all windows
- collision_features: Collision detection for all windows

Your task: For each window, combine all sensor results and compare against ODD boundaries.

Output valid JSON with this schema:
{
  "windows": [
    {
      "window_id": "000",
      "merged_features": {
        "motion": {...},
        "vision": {...},
        "terrain": {...},
        "collision": {...}
      },
      "odd_violations": ["speed_exceeded", "terrain_rough", ...],
      "overall_status": "in_odd" | "near_boundary" | "odd_exit",
      "distance_from_odd": <float 0-1>
    },
    ...
  ],
  "summary": {
    "total_windows": <int>,
    "windows_in_odd": <int>,
    "windows_near_boundary": <int>,
    "windows_odd_exit": <int>
  }
}

Evaluate all windows and return complete analysis.""",
    output_key="cod_evaluation"
)

print("✅ COD Evaluator Agent created")

### 5.7 Report Generation Agent

Creates comprehensive markdown reports with visualizations.

In [ ]:
# Report Generation Agent: Creates comprehensive markdown reports
report_agent = Agent(
    name="Report_Generator",
    model=Gemini(
        model=GEMINI_MODEL,
        api_key=GOOGLE_API_KEY
    ),
    tools=[distance_plot_tool, status_dist_tool],
    instruction="""You are a technical report writer for robotics analysis.

From the shared state, you have access to:
- odd_spec_json: The ODD specification
- cod_evaluation: Complete window-by-window COD analysis with overall summary
- motion_features, vision_features, terrain_features, collision_features: Raw sensor analysis

You have access to visualization tools:
- generate_distance_plot(times, distances, title): Returns base64 PNG
- generate_status_distribution(statuses, title): Returns base64 PNG

Generate a comprehensive markdown report including:
1. Executive Summary
   - Total windows analyzed
   - Compliance statistics (in_odd, near_boundary, odd_exit counts)
   - Overall deployment feasibility

2. Detailed Window Analysis
   - For each window: status, violations, confidence scores

3. Key Findings
   - Most critical violations
   - Patterns across windows
   - Risk assessment

4. Recommendations
   - Deployment constraints
   - Areas for improvement
   - Suggested operational limits

Output markdown text suitable for technical documentation.""",
    output_key="final_report"
)

print("✅ Report Agent created with visualization tools")

## 6. Create Parallel and Sequential Agent Workflow

Combine specialist agents using `ParallelAgent` and `SequentialAgent` following the Kaggle Day 1B pattern.

The workflow:
1. ODD Spec Agent converts NL → JSON (sequential, first)
2. Motion + Vision + Terrain + Collision agents run in parallel for each window
3. COD Evaluator aggregates results (sequential, after parallel)
4. Report Agent generates final output (sequential, last)

In [ ]:
# ParallelAgent: Run Motion, Vision, Terrain, Collision agents simultaneously
parallel_sensor_team = ParallelAgent(
    name="ParallelSensorTeam",
    sub_agents=[motion_agent, vision_agent, terrain_agent, collision_agent],
)

# SequentialAgent: Define complete workflow
# Data is pre-loaded locally, so no Data Loader Agent needed
# 1. ODD Spec Agent - converts NL to JSON spec (sequential, first)
# 2. Parallel sensor analysis team - analyzes each window (parallel)
# 3. COD Evaluator - aggregates results (sequential)
# 4. Report Agent - generates final output (sequential, last)
root_agent = SequentialAgent(
    name="ODD_COD_Analysis_System",
    sub_agents=[
        odd_spec_agent,
        parallel_sensor_team,
        cod_evaluator_agent,
        report_agent
    ],
)

print("✅ Parallel and Sequential Agents created")
print("  ParallelSensorTeam: 4 agents running simultaneously")
print("  ODD_COD_Analysis_System: 4-step sequential workflow (NO Data Loader)")
print("    1. ODD Spec Parser (NL → JSON)")
print("    2. Parallel Sensor Team (Motion, Vision, Terrain, Collision)")
print("    3. COD Evaluator (aggregates results)")
print("    4. Report Generator (final output)")

## 7. Execute the Workflow

Run the orchestrator agent with user inputs to perform complete ODD/COD analysis.

In [ ]:
# Create InMemoryRunner with the root agent
runner = InMemoryRunner(agent=root_agent)

# Execute the workflow with initial state
print("="  * 80)
print("EXECUTING ODD/COD ANALYSIS WORKFLOW")
print("=" * 80)
print(f"\nDataset: {scenario_path}")
print(f"Model: {GEMINI_MODEL}")
print(f"ODD Specification: {len(odd_natural_language)} characters\n")
print("Workflow steps:")
print("  1. ODD Spec: Convert NL → JSON")
print("  2. Parallel Sensors: Motion, Vision, Terrain, Collision")
print("  3. COD Evaluator: Aggregate against ODD")
print("  4. Report: Generate markdown report")
print(f"\n⚠️  Total: ~6-8 API calls")
print(f"  Free tier limit for {GEMINI_MODEL}: 30 RPM")
print("  Tip: Wait 60 seconds if you hit RESOURCE_EXHAUSTED, then retry")
print("-" * 80)

# Pass pre-loaded scenario_data as initial state (not as tool call)
# This avoids filesystem access from cloud agents
from google.adk.runners import InMemoryRunner

# Run the workflow using run_debug with user_messages
response_events = await runner.run_debug(user_messages=odd_natural_language)

print("\n" + "=" * 80)
print("WORKFLOW COMPLETE")
print("=" * 80)

# Process response events
if response_events:
    print(f"\n✅ Received {len(response_events)} response events")
    
    # Look for the final report in the events
    for event in response_events:
        if hasattr(event, 'data') and isinstance(event.data, dict):
            if 'final_report' in event.data:
                final_report = event.data['final_report']
                print("\n" + "=" * 80)
                print("📋 FINAL REPORT")
                print("=" * 80)
                print(final_report)
                break
    
    print("\n" + "=" * 80)
else:
    print("\n✅ Workflow executed!")
    print("(No response events returned)")

print("\n" + "=" * 80)

In [ ]:

# Extract COD Evaluation to see what data was available
print("\n" + "=" * 80)
print("COD EVALUATION RESULTS")
print("=" * 80 + "\n")

import json

for event in response_events:
    if getattr(event, 'author', '') == "COD_Evaluator":
        content = getattr(event, 'content', None)
        if content and hasattr(content, '__iter__') and not isinstance(content, str):
            for part in content:
                if hasattr(part, 'text') and part.text:
                    try:
                        # Extract JSON from markdown code block if present
                        text = part.text
                        if '```json' in text:
                            start = text.find('```json') + 7
                            end = text.find('```', start)
                            text = text[start:end].strip()
                        
                        cod_data = json.loads(text)
                        
                        # Show what the COD Evaluator saw
                        if 'windows' in cod_data and len(cod_data['windows']) > 0:
                            first_window = cod_data['windows'][0]
                            print(f"First window analysis (ID={first_window.get('window_id')}):")
                            print(json.dumps(first_window.get('merged_features', {}), indent=2))
                            
                            # Check for unknown values
                            merged = first_window.get('merged_features', {})
                            vision = merged.get('vision', {})
                            terrain = merged.get('terrain', {})
                            
                            print(f"\n❓ Vision data presence:")
                            print(f"   Keys: {list(vision.keys())}")
                            print(f"   Has 'lighting_class': {'lighting_class' in vision}")
                            print(f"   Has 'humans_visible': {'humans_visible' in vision}")
                            
                            print(f"\n❓ Terrain data presence:")
                            print(f"   Keys: {list(terrain.keys())}")
                            print(f"   Has 'terrain_roughness_class': {'terrain_roughness_class' in terrain}")
                            
                    except json.JSONDecodeError as e:
                        print(f"Could not parse COD output: {e}")



In [ ]:

# Deep dive: Check ALL events for Vision and Terrain outputs
print("\n" + "=" * 80)
print("DETAILED EVENT ANALYSIS")
print("=" * 80 + "\n")

for i, event in enumerate(response_events):
    author = getattr(event, 'author', 'Unknown')
    content = getattr(event, 'content', None)
    
    if author in ["Vision_Analyzer", "Terrain_Analyzer"]:
        print(f"\n📌 Event {i}: {author}")
        print(f"   Content type: {type(content)}")
        
        if content:
            if hasattr(content, '__iter__') and not isinstance(content, str):
                print(f"   Parts: {len(list(content))}")
                for j, part in enumerate(content):
                    print(f"     Part {j}: {type(part)}")
                    if hasattr(part, 'text'):
                        text = part.text[:200] if part.text else "(empty)"
                        print(f"       Text: {text}...")
                    if hasattr(part, 'function_call'):
                        print(f"       Function call detected")
            else:
                print(f"   Content: {str(content)[:200]}")
        else:
            print(f"   Content: None (EMPTY!)")


In [ ]:


# Extract the actual text content from Vision/Terrain events
print("\n" + "=" * 80)
print("VISION/TERRAIN JSON OUTPUTS")
print("=" * 80 + "\n")

for i, event in enumerate(response_events):
    author = getattr(event, 'author', None)
    content = getattr(event, 'content', None)
    
    if author in ["Vision_Analyzer", "Terrain_Analyzer"] and content:
        if hasattr(content, '__iter__') and not isinstance(content, str):
            for part in content:
                if isinstance(part, tuple) and len(part) > 0:
                    # Part structure: (type, value)
                    if hasattr(part, '__getitem__'):
                        try:
                            part_type = part[0] if len(part) > 0 else None
                            part_value = part[1] if len(part) > 1 else None
                            
                            if part_type == "text":
                                print(f"Event {i} - {author}:")
                                # Try to parse JSON if present
                                if part_value and '```json' in str(part_value):
                                    start = str(part_value).find('```json') + 7
                                    end = str(part_value).find('```', start)
                                    json_text = str(part_value)[start:end].strip()
                                    print(json_text[:400])
                                elif part_value and '{' in str(part_value):
                                    print(str(part_value)[:400])
                                print()
                        except:
                            pass


In [ ]:


# Print raw structure of Vision/Terrain events
print("\n" + "=" * 80)
print("RAW EVENT STRUCTURE")
print("=" * 80 + "\n")

for i, event in enumerate(response_events):
    author = getattr(event, 'author', None)
    
    if author == "Terrain_Analyzer" and i == 13:  # The final Terrain event that's empty
        print(f"Event {i} - {author} (FINAL):")
        print(f"  .content: {getattr(event, 'content', 'N/A')}")
        print(f"  .data: {getattr(event, 'data', 'N/A')}")
        print(f"  dir(event): {[x for x in dir(event) if not x.startswith('_')]}")
        break

print("\n\nLooking for JSON outputs in all events:")
for i, event in enumerate(response_events):
    author = getattr(event, 'author', None)
    if author and 'Analyzer' in author:
        # Try to get output_key
        if hasattr(event, 'output_key'):
            print(f"Event {i} - {author}: output_key = {event.output_key}")


In [ ]:


# Check the Report Generator output
print("\n" + "=" * 80)
print("REPORT GENERATOR EVENT")
print("=" * 80 + "\n")

for i, event in enumerate(response_events):
    author = getattr(event, 'author', None)
    if author == "Report_Generator":
        content = getattr(event, 'content', None)
        print(f"Event {i} - Report_Generator")
        print(f"  Content type: {type(content)}")
        
        if content and hasattr(content, '__iter__') and not isinstance(content, str):
            for j, part in enumerate(content):
                if isinstance(part, tuple) and len(part) >= 2:
                    part_type = part[0]
                    part_value = part[1]
                    if part_type == "text":
                        # Print first 1000 chars of report
                        report_text = str(part_value)
                        print(f"\n  Report text ({len(report_text)} chars):")
                        print(report_text[:1000])
                        if len(report_text) > 1000:
                            print("\n  ... [truncated] ...")


In [ ]:


# Final summary check
print("\n" + "=" * 80)
print("FINAL SUMMARY")
print("=" * 80 + "\n")

agent_outputs = {}
for i, event in enumerate(response_events):
    author = getattr(event, 'author', None)
    if author and not author.startswith('_'):
        content = getattr(event, 'content', None)
        if author not in agent_outputs:
            agent_outputs[author] = []
        agent_outputs[author].append({
            'index': i,
            'has_content': content is not None,
            'is_final': i == len(response_events) - 1
        })

for agent, events in sorted(agent_outputs.items()):
    print(f"\n{agent}:")
    for evt in events:
        status = "✅ has content" if evt['has_content'] else "❌ EMPTY"
        final = " (FINAL)" if evt['is_final'] else ""
        print(f"  Event {evt['index']}: {status}{final}")

print(f"\nTotal events: {len(response_events)}")
print(f"Total agents: {len(agent_outputs)}")
